## LLM 자동화: 보수적 규칙 필터 + Gemini 질문 분류

이 버전은 **질문 누락 최소화**를 우선합니다.

- 입력: `text_clean`
- 1차: 명백한 비질문만 규칙으로 `False / 해당없음` 처리
- 2차: 질문 가능성이 있거나 애매한 댓글은 Gemini로 전달
- 출력: `is_question`, `question_category`, `question_confidence`, `llm_error`, `classification_source`
- `classification_source`: `rule` / `gemini` / 미처리(`NA`)
- 429 / `RESOURCE_EXHAUSTED` 발생 시 즉시 체크포인트 저장 후 중단

> 규칙은 질문을 확정하지 않습니다. 규칙은 오직 **명백한 비질문**만 제외합니다.

In [1]:
# GEMINI API 설정
import json
import os
import time

from google import genai
from google.genai import types
from typing import Literal
from pydantic import BaseModel, Field

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError(
        "GEMINI_API_KEY 환경변수가 없습니다. API 키를 환경변수로 설정한 뒤 다시 실행해주세요."
    )

client = genai.Client()
MODEL_NAME = "gemini-3.1-flash-lite"

print("Gemini client 준비 완료")
print("사용 모델:", MODEL_NAME)

Gemini client 준비 완료
사용 모델: gemini-3.1-flash-lite


In [ ]:
# 고정 카테고리: 런타임 검사용 tuple과 Pydantic 타입용 Literal을 분리
QUESTION_CATEGORIES = (
    "직업_캐릭터",
    "육성_레벨링",
    "스펙업",
    "장비_아이템",
    "보스",
    "사냥",
    "스킬_6차",
    "메소마켓",
    "캐시샵",
    "경매장",
    "이벤트",
    "시스템",
    "기타",
    "해당없음",
)

QuestionCategory = Literal[
    "직업_캐릭터",
    "육성_레벨링",
    "스펙업",
    "장비_아이템",
    "보스",
    "사냥",
    "스킬_6차",
    "메소마켓",
    "캐시샵",
    "경매장",
    "이벤트",
    "시스템",
    "기타",
    "해당없음",
]


class QuestionClassification(BaseModel):
    is_question: bool
    question_category: QuestionCategory
    question_confidence: float = Field(ge=0.0, le=1.0)


print("카테고리 수:", len(QUESTION_CATEGORIES))

카테고리 수: 12


In [11]:
QUESTION_SYSTEM_PROMPT = """
너는 메이플스토리 YouTube 댓글을 분석하는 데이터 라벨러다.
각 댓글이 사용자가 정보나 조언을 얻기 위해 묻는 질문인지 판단하고,
질문이면 반드시 아래 고정 카테고리 중 하나만 선택한다.

카테고리 정의:
- 직업_캐릭터: 직업 선택, 캐릭터 추천, 직업 비교
- 육성_레벨링: 레벨업, 성장 순서, 뉴비 육성, 구간별 성장, 어떤 순서로 육성할지에 관한 질문, 특정 레벨 이후 무엇을 해야 하는지에 관한 질문
- 스펙업: 스탯, 방무, 보공, 환산, 스펙 상승 방향
- 장비_아이템: 장비를 어떻게 맞춰야 하는지에 대한 질문, 장비, 잠재, 옵션, 강화, 에테르넬, 칠흑 등
- 보스: 보스 공략, 보스 도전 스펙, 보스 관련 판단
- 사냥: 사냥터, 사냥 방식, 경험치, 사냥 효율, 메제
- 스킬_6차: 스킬, 코어, 5차/6차, 스킬 강화 순서, HEXA
- 메소마켓: 메소, 시세, 메이플포인트, 수수료, 메포
- 캐시샵: 코디템, 로얄스타일, 패스권, 펫, 마일리지, 캐시로 메포 교환하는 방법
- 경매장: 사고팔기에 적당한 가격인지에 대한 질문, 시세, 비용, 에르다조각, 아이템
- 이벤트 : 버닝, 이벤트, 서버, 게임 콘텐츠
- 시스템: 설정, 필터키, 게임 시스템, 이펙트, 소리
- 기타: 질문이지만 위 유형에 명확히 포함되지 않음
- 해당없음: 질문이 아님

카테고리 충돌 시 우선순위:
1. 시스템이 "왜 안 되는지", "어떻게 작동하는지"를 묻는 경우 -> 시스템
2. 캐릭터가 "앞으로 무엇을 해야 하는지", "어떻게 성장해야 하는지"를 묻는 경우 -> 육성_레벨링

예시:
- "260 찍었는데 이제 뭐 해야 하나요" -> 육성_레벨링
- "메인 퀘스트 스킵이 왜 안 되나요" -> 시스템_콘텐츠
- "하이퍼버닝 적용이 왜 안 되나요" -> 시스템_콘텐츠
- "하이퍼버닝 캐릭터 260 이후 뭐 해야 하나요" -> 육성_레벨링

판단 규칙:
1. 물음표가 없어도 질문 의도가 있으면 질문으로 판단한다.
2. 단순 감상, 감사, 주장, 답변, 정보 전달은 질문이 아니다.
3. 수사적 질문은 실제 정보 요청이 아니면 질문이 아니다.
4. 반드시 하나의 카테고리만 선택한다.
5. 위 우선순위와 예시를 먼저 적용한다.
6. is_question=false이면 question_category는 반드시 '해당없음'이다.
7. is_question=true이면 question_category는 '해당없음'이 될 수 없다.
8. question_confidence는 현재 판단에 대한 확신도를 0~1 사이 값으로 반환한다.
""".strip()


def validate_question_result(result):
    """Gemini 결과가 프로젝트 규칙을 만족하는지 로컬에서 재검증."""
    required = {
        "is_question",
        "question_category",
        "question_confidence",
    }

    if set(result) != required:
        raise ValueError(f"예상하지 못한 결과 필드: {set(result)}")

    category = str(result["question_category"]).strip()
    result["question_category"] = category

    if category not in QUESTION_CATEGORIES:
        raise ValueError(f"허용되지 않은 카테고리: {category!r}")

    confidence = float(result["question_confidence"])
    if not 0 <= confidence <= 1:
        raise ValueError(f"confidence 범위 오류: {confidence}")
    result["question_confidence"] = confidence

    if result["is_question"] is False:
        result["question_category"] = "해당없음"
    elif result["question_category"] == "해당없음":
        raise ValueError("질문인데 question_category가 '해당없음'으로 반환됨")

    return result


def classify_question_gemini(text_clean, max_retries=2):
    """댓글 1개를 한 번의 Gemini 호출로 질문 여부와 카테고리까지 분류."""
    text_clean = str(text_clean).strip()

    if not text_clean:
        return {
            "is_question": False,
            "question_category": "해당없음",
            "question_confidence": 1.0,
        }

    prompt = f"""
{QUESTION_SYSTEM_PROMPT}

[분석할 댓글]
{text_clean}
""".strip()

    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=QuestionClassification,
                    seed=42,
                ),
            )

            if response.parsed is not None:
                if isinstance(response.parsed, BaseModel):
                    result = response.parsed.model_dump(mode="json")
                else:
                    result = response.parsed
            else:
                result = json.loads(response.text)

            return validate_question_result(result)

        except Exception as error:
            error_message = str(error)
            last_error = error

            print(f"오류 발생 ({attempt + 1}/{max_retries})")
            print(type(error).__name__)
            print(error)

            # 하루 무료 요청량 소진 → 현재 실행에서는 해결 불가
            if (
                "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
                in error_message
            ):
                raise

            # 분당 요청량 초과
            if (
                "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
                in error_message
            ):
                match = re.search(
                    r"Please retry in ([0-9.]+)s",
                    error_message
                )

                if match:
                    wait_time = float(match.group(1)) + 2
                else:
                    wait_time = 60

                print(
                    f"RPM 제한 발생 → {wait_time:.1f}초 후 재시도"
                )

                if attempt < max_retries - 1:
                    time.sleep(wait_time)
                    continue

                raise

            # 그 외 일시적인 429
            if (
                "429" in error_message
                or "RESOURCE_EXHAUSTED" in error_message
            ):
                if attempt < max_retries - 1:
                    wait_time = 10 * (2 ** attempt)
                    print(
                        f"일시적 429 → {wait_time}초 후 재시도"
                    )
                    time.sleep(wait_time)
                    continue

                raise

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

    raise RuntimeError(
        f"LLM 분류 실패: {text_clean[:80]}"
    ) from last_error

In [12]:
# 보수적 1차 규칙 필터
# 원칙: 질문 가능성이 조금이라도 있으면 Gemini로 넘긴다.

QUESTION_SIGNAL_PATTERNS = (
    r"\?",
    r"왜|어떻게|뭐|무엇|몇|언제|어디|누구|어느",
    r"나요|가요|까요|인가요|되나요|되죠|할까요|해야|해야하나요|괜찮나요|맞나요",
    r"추천|어떤게|어느게|뭘|뭐가|가능|방법|알려|궁금",
    r"안\s*됨|안됨|도\s*됨|되\s*나|되나|돼\s*나|돼나",
)

# 댓글 전체가 아래와 사실상 동일할 때만 제외하는 매우 좁은 패턴
SHORT_NON_QUESTION_PATTERNS = (
    r"(?:ㅋ+|ㅎ+)",
    r"굿+",
    r"대박+",
    r"와+",
    r"오+",
    r"ㄷㄷ+",
    r"감사(?:합니다)?",
    r"고맙습니다",
    r"잘\s*봤습니다",
    r"잘\s*보고\s*갑니다",
    r"접수\s*(?:완료|했습니다)",
)

# 질문 신호가 없다는 전제에서만 허용하는 감사/칭찬 중심 문장
PRAISE_NON_QUESTION_PATTERNS = (
    r"^(?:영상\s*)?잘\s*봤(?:어요|습니다)[.! ]*$",
    r"^(?:영상\s*)?잘\s*보고\s*갑니다[.! ]*$",
    r"^도움\s*많이\s*됐(?:어요|습니다)[.! ]*$",
    r"^좋은\s*영상\s*감사합니다[.! ]*$",
    r"^수고하셨습니다[.! ]*$",
    r"^고생\s*많으셨(?:어요|습니다)[.! ]*$",
)


def _normalize_for_rule(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def has_question_signal(text):
    text = _normalize_for_rule(text)
    if not text:
        return False

    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in QUESTION_SIGNAL_PATTERNS
    )


def is_obvious_non_question(text):
    text = _normalize_for_rule(text)

    # 내용이 없는 댓글은 LLM 호출 가치가 없으므로 비질문 처리
    if not text:
        return True

    # 질문 가능성이 조금이라도 있으면 비질문 규칙을 적용하지 않음
    if has_question_signal(text):
        return False

    compact = re.sub(r"\s+", "", text)

    if any(
        re.fullmatch(pattern, compact, flags=re.IGNORECASE)
        for pattern in SHORT_NON_QUESTION_PATTERNS
    ):
        return True

    if any(
        re.fullmatch(pattern, text, flags=re.IGNORECASE)
        for pattern in PRAISE_NON_QUESTION_PATTERNS
    ):
        return True

    return False


def prefilter_comment(text):
    if not is_obvious_non_question(text):
        return None

    return {
        "is_question": False,
        "question_category": "해당없음",
        "question_confidence": 1.0,
    }

In [13]:
import pandas as pd
import re

# 보수적 규칙 필터 테스트: 질문 누락 방지가 최우선
protected_question_cases = [
    "260 찍었는데 이제 뭐함",
    "방무 몇줄",
    "메인퀘 스킵 왜 안됨",
    "이거 사도 됨",
    "영상 잘 봤습니다 그런데 방무 몇줄 써야 하나요",
    "어빌 이대로 써도될까요",
    "하이퍼버닝 캐릭터 뭐가 좋아요",
    # 실제 수집 댓글 회귀 사례
    "근대 지금 메이플 키워도 좋을까요?",
    "레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?",
    "템화6.7이면 전투력이 2억이나 나와요?",
    "다음 신규 보스 추가되면 수익 어떨지 궁금하네요!",
    "보스로만 번거지요? ㄷㄷ",
]

for text in protected_question_cases:
    assert has_question_signal(text) is True, text
    assert prefilter_comment(text) is None, text

obvious_non_question_cases = [
    "ㅋㅋㅋㅋ",
    "ㅎㅎㅎ",
    "굿",
    "대박",
    "감사합니다",
    "잘 봤습니다",
    "영상 잘 보고 갑니다",
    "도움 많이 됐습니다",
    "좋은 영상 감사합니다",
    "수고하셨습니다",
    "고생 많으셨어요",
]

for text in obvious_non_question_cases:
    result = prefilter_comment(text)
    assert result is not None, text
    assert result["is_question"] is False, text
    assert result["question_category"] == "해당없음", text
    assert result["question_confidence"] == 1.0, text

ambiguous_cases = [
    "저는 일단 방무 한 줄 쓰고 있습니다",
    "보스가 안 깨져서 장비를 하나씩 바꾸는 중입니다",
    "이번 패치 이후 사냥 방식이 많이 달라진 것 같네요",
    # 실제 수집 댓글 회귀 사례
    "보스가 안 깨지시거나 스펙업 수단이 없을 때만 해주시면 될 거예요!",
    "무조건 방무 바꾸는거는 아니에여 환산에서 스텟효율보고 바꾸는거에여",
]

for text in ambiguous_cases:
    assert prefilter_comment(text) is None, text

print("prefilter 테스트 통과")

prefilter 테스트 통과


In [14]:
# API smoke test는 quota가 남아 있을 때만 명시적으로 켜서 실행하세요.
RUN_API_SMOKE_TEST = False

if RUN_API_SMOKE_TEST:
    test_text = "지금 막 시작했는데 메인 퀘 다 스킵하기가 안되던데 왜 그런건가요"
    print("질문:", test_text)
    print("분류:", classify_question_gemini(test_text))
else:
    print("API smoke test 비활성화 상태입니다. quota가 가능할 때 RUN_API_SMOKE_TEST=True로 변경하세요.")

API smoke test 비활성화 상태입니다. quota가 가능할 때 RUN_API_SMOKE_TEST=True로 변경하세요.


In [15]:
# 결과 컬럼 초기화 + 체크포인트 재개
# 새 버전은 별도 checkpoint로 저장하되, 기존 checkpoint가 있으면 최초 1회 이어받습니다.
checkpoint_path = "../data/processed/youtube_question_gemini_prefilter_checkpoint.csv"
legacy_checkpoint_candidates = [
    "../data/processed/youtube_question_gemini_checkpoint.csv",
    "../data/processed/youtube_question_llm_checkpoint.csv",
]

os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

checkpoint_source = None
if os.path.exists(checkpoint_path):
    checkpoint_source = checkpoint_path
else:
    for candidate in legacy_checkpoint_candidates:
        if os.path.exists(candidate):
            checkpoint_source = candidate
            break

if checkpoint_source is not None:
    print("체크포인트를 불러옵니다:", checkpoint_source)
    df_youtube_merged_sample = pd.read_csv(checkpoint_source)

    # CSV 재로딩 시 True/False가 문자열이 되는 경우 정리
    if "is_question" in df_youtube_merged_sample.columns:
        df_youtube_merged_sample["is_question"] = (
            df_youtube_merged_sample["is_question"]
            .map({True: True, False: False, "True": True, "False": False})
            .astype("boolean")
        )

    if "question_confidence" in df_youtube_merged_sample.columns:
        df_youtube_merged_sample["question_confidence"] = pd.to_numeric(
            df_youtube_merged_sample["question_confidence"],
            errors="coerce",
        )

if "is_question" not in df_youtube_merged_sample.columns:
    df_youtube_merged_sample["is_question"] = pd.Series(
        pd.NA,
        index=df_youtube_merged_sample.index,
        dtype="boolean",
    )

for column in [
    "question_category",
    "question_confidence",
    "llm_error",
    "classification_source",
]:
    if column not in df_youtube_merged_sample.columns:
        df_youtube_merged_sample[column] = pd.NA

assert "classification_source" in df_youtube_merged_sample.columns
assert set(
    df_youtube_merged_sample["classification_source"].dropna().unique()
).issubset({"rule", "gemini"})

processed_count = df_youtube_merged_sample["is_question"].notna().sum()
print("전체 행:", len(df_youtube_merged_sample))
print("이미 처리된 행:", processed_count)
print("source 기록된 행:", df_youtube_merged_sample["classification_source"].notna().sum())

체크포인트를 불러옵니다: ../data/processed/youtube_question_gemini_prefilter_checkpoint.csv
전체 행: 2779
이미 처리된 행: 506
source 기록된 행: 25


In [16]:
# API 호출 없이 규칙 필터의 예상 효과를 먼저 확인
unprocessed_mask = df_youtube_merged_sample["is_question"].isna()

rule_preview_mask = pd.Series(
    False,
    index=df_youtube_merged_sample.index,
    dtype="boolean",
)

rule_preview_mask.loc[unprocessed_mask] = (
    df_youtube_merged_sample.loc[unprocessed_mask, "text_clean"]
    .apply(lambda text: prefilter_comment(text) is not None)
    .astype("boolean")
)

unprocessed_count = int(unprocessed_mask.sum())
rule_preview_count = int(rule_preview_mask.sum())
gemini_target_count = unprocessed_count - rule_preview_count
reduction_rate = (
    rule_preview_count / unprocessed_count * 100
    if unprocessed_count
    else 0.0
)

print("미처리 댓글:", unprocessed_count)
print("규칙으로 제외 예정:", rule_preview_count)
print("Gemini 전달 예정:", gemini_target_count)
print(f"예상 API 호출 절감률: {reduction_rate:.1f}%")

assert rule_preview_count + gemini_target_count == unprocessed_count
assert not (rule_preview_mask & ~unprocessed_mask).any()

preview_rule_rows = df_youtube_merged_sample.loc[
    rule_preview_mask,
    ["text_raw", "text_clean"],
]

if len(preview_rule_rows):
    display(
        preview_rule_rows.sample(
            min(len(preview_rule_rows), 30),
            random_state=42,
        )
    )
else:
    print("현재 규칙으로 제외 예정인 미처리 댓글이 없습니다.")

미처리 댓글: 2273
규칙으로 제외 예정: 22
Gemini 전달 예정: 2251
예상 API 호출 절감률: 1.0%


,text_raw,text_clean
629,굿,굿
2035,.,NaN
1419,👏🏻👏🏻👏🏻,NaN
837,❤,NaN
2196,감사합니다,감사합니다
1215,😮,NaN
2525,감사합니다!,감사합니다
1867,.,NaN
1122,ㄷㄷ,ㄷㄷ
1213,😮,NaN


In [17]:
# 전체 댓글 분류: 보수적 규칙 필터 -> Gemini
# 429 발생 시 즉시 저장 후 중단하며, 다음 실행에서 미처리 행부터 재개합니다.
REQUEST_DELAY = 5
checkpoint_every = 50
processed_since_save = 0
api_request_count = 0
rule_processed_count = 0

for idx, row in df_youtube_merged_sample.iterrows():
    # 기존 완료 행은 어떤 방식으로 처리됐든 그대로 보존
    if pd.notna(row["is_question"]):
        continue

    rule_result = prefilter_comment(row["text_clean"])

    if rule_result is not None:
        df_youtube_merged_sample.at[idx, "is_question"] = rule_result["is_question"]
        df_youtube_merged_sample.at[idx, "question_category"] = rule_result["question_category"]
        df_youtube_merged_sample.at[idx, "question_confidence"] = rule_result["question_confidence"]
        df_youtube_merged_sample.at[idx, "llm_error"] = pd.NA
        df_youtube_merged_sample.at[idx, "classification_source"] = "rule"
        rule_processed_count += 1

    else:
        try:
            result = classify_question_gemini(row["text_clean"])

            df_youtube_merged_sample.at[idx, "is_question"] = result["is_question"]
            df_youtube_merged_sample.at[idx, "question_category"] = result["question_category"]
            df_youtube_merged_sample.at[idx, "question_confidence"] = result["question_confidence"]
            df_youtube_merged_sample.at[idx, "llm_error"] = pd.NA
            df_youtube_merged_sample.at[idx, "classification_source"] = "gemini"
            api_request_count += 1

        except Exception as error:
            error_message = str(error)
            df_youtube_merged_sample.at[idx, "llm_error"] = error_message

            if (
                "429" in error_message
                or "RESOURCE_EXHAUSTED" in error_message
            ):
                df_youtube_merged_sample.to_csv(
                    checkpoint_path,
                    index=False,
                    encoding="utf-8-sig",
                )
                print(f"Quota 초과 -> index={idx}까지 저장 후 중단")
                break

            print(f"index={idx} 처리 실패: {error_message}")

        # Gemini 경로에서만 다음 요청 전 대기
        time.sleep(REQUEST_DELAY)

    processed_since_save += 1

    if processed_since_save >= checkpoint_every:
        df_youtube_merged_sample.to_csv(
            checkpoint_path,
            index=False,
            encoding="utf-8-sig",
        )
        processed_since_save = 0
        print(f"checkpoint 저장 완료: index={idx}")

# 루프 종료 사유와 관계없이 마지막 상태 저장
df_youtube_merged_sample.to_csv(
    checkpoint_path,
    index=False,
    encoding="utf-8-sig",
)

print("현재까지 결과 저장:", checkpoint_path)
print("이번 실행 rule 처리:", rule_processed_count)
print("이번 실행 Gemini 성공 요청:", api_request_count)

checkpoint 저장 완료: index=555
checkpoint 저장 완료: index=605
checkpoint 저장 완료: index=655
checkpoint 저장 완료: index=705
checkpoint 저장 완료: index=755
checkpoint 저장 완료: index=805
checkpoint 저장 완료: index=855
checkpoint 저장 완료: index=905
checkpoint 저장 완료: index=955
오류 발생 (1/2)
ClientError
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 31.728497526s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.

In [18]:
# 결과 정합성 검증 + 질문 댓글 추출
invalid_false = df_youtube_merged_sample[
    (df_youtube_merged_sample["is_question"] == False)
    & (df_youtube_merged_sample["question_category"] != "해당없음")
]

invalid_true = df_youtube_merged_sample[
    (df_youtube_merged_sample["is_question"] == True)
    & (df_youtube_merged_sample["question_category"] == "해당없음")
]

assert invalid_false.empty, "질문이 아닌데 카테고리가 지정된 행이 있습니다."
assert invalid_true.empty, "질문인데 question_category가 '해당없음'인 행이 있습니다."

rule_rows = df_youtube_merged_sample[
    df_youtube_merged_sample["classification_source"] == "rule"
]

assert (rule_rows["is_question"] == False).all()
assert (rule_rows["question_category"] == "해당없음").all()
assert (rule_rows["question_confidence"] == 1.0).all()
assert rule_rows["llm_error"].isna().all()

assert set(
    df_youtube_merged_sample["classification_source"].dropna().unique()
).issubset({"rule", "gemini"})

print("LLM 오류 수:", df_youtube_merged_sample["llm_error"].notna().sum())
print("질문 수:", (df_youtube_merged_sample["is_question"] == True).sum())
print("질문 비율:", (df_youtube_merged_sample["is_question"] == True).mean())
print()

print("분류 출처")
display(
    df_youtube_merged_sample["classification_source"]
    .value_counts(dropna=False)
    .rename_axis("classification_source")
    .reset_index(name="count")
)

print("confidence 요약")
display(df_youtube_merged_sample["question_confidence"].describe())

print("카테고리 빈도")
display(
    df_youtube_merged_sample["question_category"]
    .value_counts(dropna=False)
    .rename_axis("question_category")
    .reset_index(name="count")
)

df_youtube_questions = df_youtube_merged_sample[
    df_youtube_merged_sample["is_question"] == True
].copy()

question_save_path = "../data/processed/youtube_questions_gemini_prefilter.csv"
df_youtube_questions.to_csv(
    question_save_path,
    index=False,
    encoding="utf-8-sig",
)

print("질문 데이터 저장:", question_save_path)

LLM 오류 수: 1
질문 수: 344
질문 비율: 0.3524590163934426

분류 출처


,classification_source,count
0,NaN,2284
1,gemini,492
2,rule,3


confidence 요약


count    976.000000
mean       0.976844
std        0.039635
min        0.800000
25%        0.950000
50%        1.000000
75%        1.000000
max        1.000000
Name: question_confidence, dtype: float64

카테고리 빈도


,question_category,count
0,NaN,1803
1,해당없음,632
2,기타,55
3,육성_레벨링,52
4,보스,51
5,장비_아이템,41
6,시스템,40
7,이벤트,32
8,사냥,27
9,스펙업,21


질문 데이터 저장: ../data/processed/youtube_questions_gemini_prefilter.csv


In [19]:
# rule로 제외된 행 최대 100건 수동 검수
rule_rows = df_youtube_merged_sample[
    df_youtube_merged_sample["classification_source"] == "rule"
]

if len(rule_rows):
    rule_review_sample = rule_rows.sample(
        min(len(rule_rows), 100),
        random_state=42,
    )

    display(
        rule_review_sample[
            [
                "text_raw",
                "text_clean",
                "is_question",
                "question_category",
                "classification_source",
            ]
        ]
    )
else:
    print("rule 처리된 행이 없습니다.")

,text_raw,text_clean,is_question,question_category,classification_source
629,굿,굿,False,해당없음,rule
837,❤,NaN,False,해당없음,rule
928,와😊,와,False,해당없음,rule


In [20]:
# 카테고리별 최대 10건씩 사람이 직접 샘플 검수
classified_rows = df_youtube_merged_sample.dropna(subset=["question_category"])

review_parts = []
for _, group in classified_rows.groupby("question_category"):
    review_parts.append(
        group.sample(min(len(group), 10), random_state=42)
    )

if review_parts:
    review_sample = pd.concat(review_parts).sort_index()
else:
    review_sample = classified_rows.head(0).copy()

display(
    review_sample[
        [
            "text_raw",
            "text_clean",
            "is_question",
            "question_category",
            "question_confidence",
            "classification_source",
        ]
    ]
)

,text_raw,text_clean,is_question,question_category,question_confidence,classification_source
6,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,레테스킬에는 방무가 있던데 그래도 따로 방무가필요할까요?,True,스펙업,1.00,NaN
11,템화6.7이면 전투력이 2억이나 나와요?\r\n\r\n패파라서그런가 템환6.7로 1...,템화6 7이면 전투력이 2억이나 나와요? 패파라서그런가 템환6 7로 1 4억이던데,True,스펙업,0.95,NaN
13,내실도 해줘,내실도 해줘,True,육성_레벨링,0.90,NaN
25,1주차라는게 영상업로드 기준인가요 아니면 이번 목요일부터 시작되는건가요!!? 유입 ...,1주차라는게 영상업로드 기준인가요 아니면 이번 목요일부터 시작되는건가요 ? 유입 4...,True,이벤트,1.00,NaN
33,맑음님 정도 스팩 이면 상위 10위권 되나? 퍼클 에도 여러번 이름 올리니까,맑음님 정도 스팩 이면 상위 10위권 되나? 퍼클 에도 여러번 이름 올리니까,True,스펙업,0.90,NaN
...,...,...,...,...,...,...
943,그니까 에픽던전 1단계 할 돈으로 쌀팔이 해서 모멘텀 사는 게 나을 정도인 거죠?,그니까 에픽던전 1단계 할 돈으로 쌀팔이 해서 모멘텀 사는 게 나을 정도인 거죠?,True,재화_경제,0.90,gemini
958,"혹시 저만 중복떠서 법사 3등급 다 맞춰주고 전사랑 궁수는 2,3 등급 섞여있는데 ...",혹시 저만 중복떠서 법사 3등급 다 맞춰주고 전사랑 궁수는 2 3 등급 섞여있는데 ...,True,장비_아이템,0.95,gemini
969,전사 피격시 무적은 별로인가여?,전사 피격시 무적은 별로인가여?,True,스킬_6차,0.95,gemini
970,전사 스킬은 뭐쓰나요?,전사 스킬은 뭐쓰나요?,True,스킬_6차,0.95,gemini
